# Movie Recommendation System using MovieLens

## Content-Based Recommender with Feature Engineering and Popularity-Aware Ranking

**Author:** Dmitry  
**Project Type:** Portfolio Project  
**Dataset:** MovieLens Latest Small  
**Goal:** Build a movie recommendation system that suggests similar movies based on title, genres, and user-generated tags.

## 1. Business Problem

Recommendation systems help users discover relevant content in large catalogs.  
In the movie industry, a recommendation engine can improve user engagement by suggesting films similar to those a user already likes.

In this project, I build a **content-based recommendation system** using the MovieLens dataset.  
The system recommends movies based on:

- movie title
- genres
- user-generated tags

To improve practical usefulness, recommendations are ranked using both:

- content similarity
- movie quality and popularity

## 2. Project Objectives

The main objectives of this project are:

1. Load and clean the MovieLens dataset
2. Engineer useful features from raw movie data
3. Create text-based movie representations
4. Compute similarity between movies
5. Build a recommendation function
6. Improve ranking with weighted ratings and popularity
7. Produce a portfolio-ready recommender system

## 3. Import Libraries

First, import the libraries needed for:

- data manipulation
- text preprocessing
- similarity calculation

In [6]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 4. Load the Dataset

This project uses three files from the MovieLens dataset:

- `movies.csv` — movie titles and genres
- `ratings.csv` — user ratings
- `tags.csv` — user-generated descriptive tags

In [8]:
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")
tags = pd.read_csv("data/tags.csv")

## 5. Initial Data Inspection

Before building the recommender, inspect the shape and structure of each dataset.

In [10]:
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
print("Tags shape:", tags.shape)

print("\nMovies columns:", movies.columns.tolist())
print("Ratings columns:", ratings.columns.tolist())
print("Tags columns:", tags.columns.tolist())

Movies shape: (9742, 3)
Ratings shape: (100836, 4)
Tags shape: (3683, 4)

Movies columns: ['movieId', 'title', 'genres']
Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']
Tags columns: ['userId', 'movieId', 'tag', 'timestamp']


In [11]:
print("Movies sample:")
display(movies.head())

print("Ratings sample:")
display(ratings.head())

print("Tags sample:")
display(tags.head())

Movies sample:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings sample:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Tags sample:


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


## 6. Data Cleaning

The raw data may contain missing values in text columns.  
These missing values are filled to avoid errors during feature engineering.

In [12]:
movies["title"] = movies["title"].fillna("")
movies["genres"] = movies["genres"].fillna("")
tags["tag"] = tags["tag"].fillna("")

## 7. Feature Engineering: Extract Release Year

Movie titles in MovieLens usually contain the release year in parentheses, such as:

- `Toy Story (1995)`

Extracting the year gives an additional structured feature that can be useful for analysis.

In [13]:
def extract_year(title):
    match = re.search(r"\((\d{4})\)", title)
    return int(match.group(1)) if match else np.nan

movies["year"] = movies["title"].apply(extract_year)

In [14]:
display(movies[["title", "year"]].head())

,title,year
0,Toy Story (1995),1995.0
1,Jumanji (1995),1995.0
2,Grumpier Old Men (1995),1995.0
3,Waiting to Exhale (1995),1995.0
4,Father of the Bride Part II (1995),1995.0


## 8. Feature Engineering: Rating Statistics

To make recommendations more practical, create rating-based features for each movie:

- `num_ratings` — how many ratings the movie received
- `avg_rating` — the average rating
- `rating_std` — how much ratings vary

In [15]:
rating_stats = ratings.groupby("movieId").agg(
    num_ratings=("rating", "count"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std")
).reset_index()

In [16]:
display(rating_stats.head())

,movieId,num_ratings,avg_rating,rating_std
0,1,215,3.920930,0.834859
1,2,110,3.431818,0.881713
2,3,52,3.259615,1.054823
3,4,7,2.357143,0.852168
4,5,49,3.071429,0.907148


Merge these rating statistics into the main movie table.

In [18]:
movies = movies.merge(rating_stats, on="movieId", how="left")

In [19]:
movies["num_ratings"] = movies["num_ratings"].fillna(0)
movies["avg_rating"] = movies["avg_rating"].fillna(0)
movies["rating_std"] = movies["rating_std"].fillna(0)

## 9. Feature Engineering: User Tags

The `tags.csv` file contains user-generated descriptors such as:

- funny
- action
- sci-fi
- emotional

These tags provide richer text information than genres alone.  
Aggregate all tags for each movie into a single text field.

In [20]:
tag_text = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.astype(str))).reset_index()
tag_text.rename(columns={"tag": "all_tags"}, inplace=True)

In [21]:
display(tag_text.head())

,movieId,all_tags
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


Now merge the aggregated tags into the movie dataset.

In [22]:
movies = movies.merge(tag_text, on="movieId", how="left")
movies["all_tags"] = movies["all_tags"].fillna("")

## 10. Clean Movie Titles

For text modeling, it is useful to remove the year from the title and convert the title to lowercase.

Example:

- `Toy Story (1995)` → `toy story`

In [23]:
def clean_title(title):
    return re.sub(r"\(\d{4}\)", "", title).strip().lower()

movies["clean_title"] = movies["title"].apply(clean_title)

In [24]:
display(movies[["title", "clean_title"]].head())

,title,clean_title
0,Toy Story (1995),toy story
1,Jumanji (1995),jumanji
2,Grumpier Old Men (1995),grumpier old men
3,Waiting to Exhale (1995),waiting to exhale
4,Father of the Bride Part II (1995),father of the bride part ii


## 11. Build Combined Text Features

To create stronger movie representations, combine:

- cleaned title
- genres
- user tags

This gives a richer input for text vectorization.

In [25]:
movies["combined_features"] = (
    movies["clean_title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["all_tags"]
)

In [26]:
display(movies[["title", "combined_features"]].head())

,title,combined_features
0,Toy Story (1995),toy story Adventure Animation Children Comedy ...
1,Jumanji (1995),jumanji Adventure Children Fantasy fantasy mag...
2,Grumpier Old Men (1995),grumpier old men Comedy Romance moldy old
3,Waiting to Exhale (1995),waiting to exhale Comedy Drama Romance
4,Father of the Bride Part II (1995),father of the bride part ii Comedy pregnancy r...


## 12. Create a Weighted Rating

A movie with a very small number of ratings can have an unreliable average score.  
To reduce this problem, create a weighted rating inspired by the IMDb approach.

This gives more reliable ranking by balancing:

- average movie rating
- number of ratings
- overall dataset average

In [27]:
C = movies["avg_rating"].mean()
m = movies["num_ratings"].quantile(0.75)

movies["weighted_rating"] = (
    (movies["num_ratings"] / (movies["num_ratings"] + m)) * movies["avg_rating"] +
    (m / (movies["num_ratings"] + m)) * C
)

In [28]:
display(movies[["title", "avg_rating", "num_ratings", "weighted_rating"]].head())

,title,avg_rating,num_ratings,weighted_rating
0,Toy Story (1995),3.920930,215.0,3.894231
1,Jumanji (1995),3.431818,110.0,3.418553
2,Grumpier Old Men (1995),3.259615,52.0,3.259144
3,Waiting to Exhale (1995),2.357143,7.0,2.862986
4,Father of the Bride Part II (1995),3.071429,49.0,3.100134


## 13. Exploratory Data Analysis

Before building the model, review the dataset at a high level.

In [29]:
print("Number of unique movies:", movies["movieId"].nunique())
print("Number of unique users:", ratings["userId"].nunique())
print("Number of ratings:", len(ratings))

Number of unique movies: 9742
Number of unique users: 610
Number of ratings: 100836


In [30]:
movies[["num_ratings", "avg_rating", "weighted_rating"]].describe()

,num_ratings,avg_rating,weighted_rating
count,9742.000000,9742.000000,9742.000000
mean,10.350647,3.256420,3.281654
std,22.384729,0.880292,0.242327
min,0.000000,0.000000,2.135992
25%,1.000000,2.785714,3.139060
50%,3.000000,3.416667,3.280778
75%,9.000000,3.909091,3.400432
max,329.000000,5.000000,4.396650


In [31]:
top_popular = movies.sort_values("num_ratings", ascending=False)[
    ["title", "num_ratings", "avg_rating", "weighted_rating"]
].head(10)

display(top_popular)

,title,num_ratings,avg_rating,weighted_rating
314,Forrest Gump (1994),329.0,4.164134,4.139964
277,"Shawshank Redemption, The (1994)",317.0,4.429022,4.396650
257,Pulp Fiction (1994),307.0,4.197068,4.170278
510,"Silence of the Lambs, The (1991)",279.0,4.161290,4.133013
1939,"Matrix, The (1999)",278.0,4.192446,4.163093
224,Star Wars: Episode IV - A New Hope (1977),251.0,4.231076,4.197338
418,Jurassic Park (1993),238.0,3.750000,3.732015
97,Braveheart (1995),237.0,4.031646,4.003284
507,Terminator 2: Judgment Day (1991),224.0,3.970982,3.943381
461,Schindler's List (1993),220.0,4.225000,4.186934


## 14. Convert Text into Numerical Features

Machine learning models cannot use raw text directly.  
Apply **TF-IDF vectorization** to transform the combined text into numerical features.

TF-IDF helps identify important words while reducing the impact of very common words.

In [33]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["combined_features"])

In [34]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9742, 9860)


## 15. Compute Movie Similarity

Use **cosine similarity** to measure how similar each movie is to every other movie based on the TF-IDF features.

In [35]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [36]:
print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (9742, 9742)


## 16. Create a Movie Title Index

To retrieve movie rows efficiently, create an index that maps each movie title to its row number.

In [37]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

In [38]:
display(indices.head())

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

## 17. Create a Helper Function to Search Movie Titles

Movie titles must usually match exactly.  
This helper function makes it easier to find the correct title before generating recommendations.

In [39]:
def find_movie(partial_title, n=10):
    matches = movies[movies["title"].str.contains(partial_title, case=False, na=False)]
    return matches[["title", "genres", "year", "avg_rating", "num_ratings"]].head(n)

In [40]:
find_movie("toy story")

,title,genres,year,avg_rating,num_ratings
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,3.920930,215.0
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1999.0,3.860825,97.0
7355,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010.0,4.109091,55.0


## 18. Build the Recommendation Function

The recommendation function will:

1. find the selected movie
2. retrieve the most similar movies
3. exclude the original movie
4. filter out weak candidates with too few ratings
5. rank recommendations using:
   - similarity
   - weighted rating
   - popularity

In [41]:
def recommend_movies(title, top_n=10, min_ratings=20):
    if title not in indices:
        return f"Movie '{title}' not found. Use find_movie() first."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:50]

    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    recs = movies.iloc[movie_indices][[
        "title", "genres", "year", "avg_rating", "num_ratings", "weighted_rating"
    ]].copy()

    recs["similarity"] = similarity_values
    recs = recs[recs["num_ratings"] >= min_ratings]

    if len(recs) == 0:
        return "No recommendations found with the current filters."

    max_log_votes = np.log1p(recs["num_ratings"].max())

    recs["final_score"] = (
        recs["similarity"] * 0.6 +
        (recs["weighted_rating"] / 5.0) * 0.3 +
        (np.log1p(recs["num_ratings"]) / max_log_votes) * 0.1
    )

    recs = recs.sort_values("final_score", ascending=False).head(top_n)

    return recs[[
        "title", "year", "genres", "avg_rating", "num_ratings",
        "weighted_rating", "similarity", "final_score"
    ]].reset_index(drop=True)

## 19. Create a Cleaner Output Function

This function prints recommendations in a readable format.

In [42]:
def print_recommendations(title, top_n=10, min_ratings=20):
    result = recommend_movies(title, top_n=top_n, min_ratings=min_ratings)

    if isinstance(result, str):
        print(result)
        return

    print(f"\nIf you liked '{title}', you might also like:\n")
    for i, row in enumerate(result.itertuples(), start=1):
        year = int(row.year) if pd.notnull(row.year) else "N/A"
        print(
            f"{i}. {row.title} ({year})\n"
            f"   Genres: {row.genres}\n"
            f"   Avg rating: {row.avg_rating:.2f} | Ratings count: {int(row.num_ratings)}\n"
            f"   Weighted rating: {row.weighted_rating:.2f} | Similarity: {row.similarity:.3f} | Final score: {row.final_score:.3f}\n"
        )

## 20. Test the Recommendation System

First, search for a movie title.  
Then, use the exact title to generate recommendations.

In [43]:
find_movie("toy story")
find_movie("matrix")
find_movie("titanic")

,title,genres,year,avg_rating,num_ratings
1291,Titanic (1997),Drama|Romance,1997.0,3.414286,140.0
2542,Raise the Titanic (1980),Drama|Thriller,1980.0,4.000000,1.0
2543,Titanic (1953),Action|Drama,1953.0,3.583333,6.0
3553,Titanica (1992),Documentary|IMAX,1992.0,2.500000,1.0


In [44]:
print_recommendations("Titanic (1997)")


If you liked 'Titanic (1997)', you might also like:

1. She's All That (1999) (1999)
   Genres: Comedy|Romance
   Avg rating: 2.84 | Ratings count: 29
   Weighted rating: 2.94 | Similarity: 0.327 | Final score: 0.473

2. Her (2013) (2013)
   Genres: Drama|Romance|Sci-Fi
   Avg rating: 3.92 | Ratings count: 25
   Weighted rating: 3.74 | Similarity: 0.235 | Final score: 0.462



## 21. Bonus: Recommend Top Movies by Genre

As an additional feature, recommend high-quality movies within a selected genre.

In [46]:
def recommend_by_genre(genre, top_n=10, min_ratings=50):
    filtered = movies[
        movies["genres"].str.contains(genre, case=False, na=False) &
        (movies["num_ratings"] >= min_ratings)
    ].copy()

    filtered = filtered.sort_values(["weighted_rating", "num_ratings"], ascending=False)

    return filtered[["title", "genres", "avg_rating", "num_ratings", "weighted_rating"]].head(top_n).reset_index(drop=True)

In [47]:
recommend_by_genre("Comedy")

,title,genres,avg_rating,num_ratings,weighted_rating
0,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.268041,97.0,4.182149
1,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.232394,142.0,4.174224
2,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.197068,307.0,4.170278
3,Forrest Gump (1994),Comedy|Drama|Romance|War,4.164134,329.0,4.139964
4,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.183333,120.0,4.118665
5,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy,4.161765,136.0,4.105571
6,Snatch (2000),Comedy|Crime|Thriller,4.155914,93.0,4.076547
7,Fargo (1996),Comedy|Crime|Drama|Thriller,4.116022,181.0,4.075304
8,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.147727,88.0,4.065029
9,Office Space (1999),Comedy|Crime,4.090426,94.0,4.017551


## 22. Save the Enriched Dataset

Save the improved movie dataset for future use or deployment.

In [48]:
movies.to_csv("movies_enriched.csv", index=False)

## 23. Conclusion

In this project, I built a **content-based movie recommendation system** using the MovieLens dataset.

### Key improvements included:
- cleaning and preparing raw movie, rating, and tag data
- extracting release year from titles
- engineering rating-based features
- aggregating user tags
- creating combined text features
- vectorizing movie text with TF-IDF
- computing movie similarity using cosine similarity
- improving ranking with weighted rating and popularity

### Result
The final system recommends movies that are both:
- textually similar to the selected title
- practically stronger based on quality and popularity

### Future Improvements
Possible next steps include:
- collaborative filtering
- hybrid recommendation systems
- movie description embeddings
- deployment with Streamlit

## 24. Project Summary

This project demonstrates how to build a professional recommendation system pipeline from raw data to ranked output.

**Skills demonstrated:**
- Pandas data cleaning
- feature engineering
- text vectorization with TF-IDF
- similarity modeling
- popularity-aware ranking
- recommendation system design

**Tools used:**
- Python
- Pandas
- NumPy
- scikit-learn